# 🏦 Phân loại Intent Ngân hàng với Unsloth

Fine-tune model **Llama-3.2-3B-Instruct** để phát hiện ý định (intent) trên dataset BANKING77.

- **Model:** `unsloth/Llama-3.2-3B-Instruct` (QLoRA 4-bit)
- **Dataset:** BANKING77 (45 intents)
- **Framework:** Unsloth + SFTTrainer
- **Môi trường:** Google Colab Free (T4 GPU)

---

## 1. Cài đặt thư viện

In [1]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install datasets scikit-learn pyyaml

## 2. Mount Google Drive (để lưu checkpoint)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/banking-intent-unsloth/outputs"
!mkdir -p {SAVE_DIR}

Mounted at /content/drive


## 3. Cấu hình

In [ ]:
CONFIG = {
    "model_name": "unsloth/Llama-3.2-3B-Instruct",
    "max_seq_length": 256,
    "load_in_4bit": True,

    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.0,
    "target_modules": [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],

    "num_epochs": 4,
    "per_device_train_batch_size": 8,
    "gradient_accumulation_steps": 4,
    "learning_rate": 2e-4,
    "weight_decay": 0.01,
    "warmup_steps": 15,
    "optimizer": "adamw_8bit",
    "lr_scheduler_type": "cosine",
    "seed": 42,
    "logging_steps": 10,

    "dataset_name": "PolyAI/banking77",
    "num_intents": 45,
    "test_size": 0.2,
}

print("Config loaded!")

Config loaded!


## 4. Tiền xử lý dữ liệu

In [ ]:
import json
import random

import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split

PROMPT_TEMPLATE = (
    "Below is a customer message sent to a bank's support system. "
    "Classify the message into exactly one intent category. "
    "Output only the intent label, nothing else.\n\n"
    "### Message:\n{message}\n\n"
    "### Intent:\n{label}"
)

INFERENCE_TEMPLATE = (
    "Below is a customer message sent to a bank's support system. "
    "Classify the message into exactly one intent category. "
    "Output only the intent label, nothing else.\n\n"
    "### Message:\n{message}\n\n"
    "### Intent:\n"
)

print("Loading BANKING77 dataset...")
dataset = load_dataset(CONFIG["dataset_name"], revision="refs/convert/parquet")
all_intent_names = dict(enumerate(dataset["train"].features["label"].names))
print(f"Total intents: {len(all_intent_names)}")

random.seed(CONFIG["seed"])
sampled_ids = sorted(random.sample(list(all_intent_names.keys()), CONFIG["num_intents"]))
sampled_names = {i: all_intent_names[i] for i in sampled_ids}
print(f"\nSelected {len(sampled_ids)} intents:")
for idx, (k, v) in enumerate(sampled_names.items()):
    print(f"  [{idx:2d}] {v}")

new_label_map = {}
old_to_new = {}
for new_id, old_id in enumerate(sampled_ids):
    new_label_map[new_id] = all_intent_names[old_id]
    old_to_new[old_id] = new_id

with open("label_map.json", "w") as f:
    json.dump(new_label_map, f, indent=2)
print(f"\nLabel map saved")

Loading BANKING77 dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


0000.parquet:   0%|          | 0.00/298k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/93.9k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Total intents: 77

Selected 45 intents:
  [ 0] activate_my_card
  [ 1] age_limit
  [ 2] atm_support
  [ 3] automatic_top_up
  [ 4] balance_not_updated_after_bank_transfer
  [ 5] balance_not_updated_after_cheque_or_cash_deposit
  [ 6] card_about_to_expire
  [ 7] card_acceptance
  [ 8] card_arrival
  [ 9] card_delivery_estimate
  [10] card_linking
  [11] card_not_working
  [12] card_payment_not_recognised
  [13] card_payment_wrong_exchange_rate
  [14] change_pin
  [15] compromised_card
  [16] country_support
  [17] declined_cash_withdrawal
  [18] declined_transfer
  [19] direct_debit_payment_not_recognised
  [20] exchange_charge
  [21] exchange_rate
  [22] extra_charge_on_statement
  [23] failed_transfer
  [24] get_disposable_virtual_card
  [25] get_physical_card
  [26] lost_or_stolen_card
  [27] order_physical_card
  [28] passcode_forgotten
  [29] pending_card_payment
  [30] pending_transfer
  [31] reverted_card_payment?
  [32] supported_cards_and_currencies
  [33] top_up_limits
  [34] 

In [ ]:
def preprocess_text(text):
    text = text.strip()
    if text and text[0].islower():
        text = text[0].upper() + text[1:]
    if text and text[-1] not in ".?!":
        text += "."
    return text

EOS_TOKEN = "</s>"

sampled_id_set = set(sampled_ids)
formatted = []

for split in ["train", "test"]:
    for example in dataset[split]:
        if example["label"] in sampled_id_set:
            message = preprocess_text(example["text"])
            label_name = all_intent_names[example["label"]]
            text = PROMPT_TEMPLATE.format(message=message, label=label_name) + EOS_TOKEN
            formatted.append({
                "text": text,
                "label": old_to_new[example["label"]],
                "label_name": label_name,
                "original_text": example["text"],
            })

df = pd.DataFrame(formatted)
print(f"Total formatted samples: {len(df)}")
print(f"\nSample:")
print(df.iloc[0]["text"])


Total formatted samples: 7595

Sample:
Below is a customer message sent to a bank's support system. Classify the message into exactly one intent category. Output only the intent label, nothing else.

### Message:
I am still waiting on my card?

### Intent:
card_arrival</s>


In [ ]:
train_df, test_df = train_test_split(
    df, test_size=CONFIG["test_size"], random_state=CONFIG["seed"], stratify=df["label"],
)
train_df.to_csv("train.csv", index=False)
test_df.to_csv("test.csv", index=False)
print(f"Train: {len(train_df)} | Test: {len(test_df)}")

Train: 6076 | Test: 1519


## 5. Tải Model

In [ ]:
from unsloth import FastLanguageModel

print(f"Loading model: {CONFIG['model_name']}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    load_in_4bit=CONFIG["load_in_4bit"],
    dtype=None,
)

EOS_TOKEN = tokenizer.eos_token
print(f"EOS token: {repr(EOS_TOKEN)}")

print("Re-formatting data with correct EOS token...")
formatted = []
sampled_id_set = set(sampled_ids)
for split in ["train", "test"]:
    for example in dataset[split]:
        if example["label"] in sampled_id_set:
            message = preprocess_text(example["text"])
            label_name = all_intent_names[example["label"]]
            text = PROMPT_TEMPLATE.format(message=message, label=label_name) + EOS_TOKEN
            formatted.append({
                "text": text,
                "label": old_to_new[example["label"]],
                "label_name": label_name,
                "original_text": example["text"],
            })
df = pd.DataFrame(formatted)
train_df, test_df = train_test_split(
    df, test_size=CONFIG["test_size"], random_state=CONFIG["seed"], stratify=df["label"],
)
train_df.to_csv("train.csv", index=False)
test_df.to_csv("test.csv", index=False)
print(f"Data re-formatted. Train: {len(train_df)} | Test: {len(test_df)}")

print("Model loaded!")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading model: unsloth/Llama-3.2-3B-Instruct...
==((====))==  Unsloth 2026.4.6: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


EOS token: '<|eot_id|>'
Re-formatting data with correct EOS token...
Data re-formatted. Train: 6076 | Test: 1519
Model loaded!


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    target_modules=CONFIG["target_modules"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=CONFIG["seed"],
)

print(f"LoRA configured (r={CONFIG['lora_r']}, alpha={CONFIG['lora_alpha']})")
model.print_trainable_parameters()

Unsloth 2026.4.6 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


LoRA configured (r=16, alpha=32)
trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


## 6. Huấn luyện

In [ ]:
from datasets import Dataset as HFDataset
from trl import SFTTrainer
from transformers import TrainingArguments

train_dataset = HFDataset.from_pandas(train_df)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

training_args = TrainingArguments(
    output_dir=SAVE_DIR,
    num_train_epochs=CONFIG["num_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
    warmup_steps=CONFIG["warmup_steps"],
    optim=CONFIG["optimizer"],
    lr_scheduler_type=CONFIG["lr_scheduler_type"],
    seed=CONFIG["seed"],
    fp16=False,
    bf16=False,
    logging_steps=CONFIG["logging_steps"],
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
)

def formatting_func(examples):
    return examples["text"]

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    args=training_args,
    formatting_func=formatting_func,
    max_seq_length=CONFIG["max_seq_length"],
    packing=False,
)



Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/6076 [00:00<?, ? examples/s]

Trainer ready. Starting training...


In [10]:
# Train!
trainer_stats = trainer.train()
print(f"\nTraining complete!")
print(f"  Total steps: {trainer_stats.global_step}")
print(f"  Training loss: {trainer_stats.training_loss:.4f}")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6,076 | Num Epochs = 4 | Total steps = 760
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,3.103878
20,1.022740
30,0.689556
40,0.631171
50,0.585737
60,0.540950
70,0.509673
80,0.516312
90,0.552490
100,0.511284



Training complete!
  Total steps: 760
  Training loss: 0.4530


## 7. Lưu Model

In [ ]:
import shutil

LOCAL_SAVE = "./banking-intent-model"
model.save_pretrained(LOCAL_SAVE)
tokenizer.save_pretrained(LOCAL_SAVE)
print(f"Model saved locally to {LOCAL_SAVE}")

shutil.copy("label_map.json", f"{LOCAL_SAVE}/label_map.json")

!cp -r {LOCAL_SAVE}/* {SAVE_DIR}/
!cp label_map.json {SAVE_DIR}/
!cp train.csv {SAVE_DIR}/
!cp test.csv {SAVE_DIR}/
print(f"Model copied to Google Drive: {SAVE_DIR}")

Model saved locally to ./banking-intent-model
Model copied to Google Drive: /content/drive/MyDrive/banking-intent-unsloth/outputs


## 8. Đánh giá

In [ ]:
import torch
from sklearn.metrics import accuracy_score, classification_report

FastLanguageModel.for_inference(model)

true_labels = []
pred_labels = []
correct = 0
total = len(test_df)

print(f"Evaluating on {total} test samples...\n")

for idx, row in test_df.iterrows():
    original_text = row["original_text"]
    true_label_name = row["label_name"]
    true_labels.append(true_label_name)

    prompt = INFERENCE_TEMPLATE.format(message=original_text)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=16, 
            do_sample=False,
            use_cache=True,
        )

    prompt_length = inputs["input_ids"].shape[1]
    new_tokens = outputs[0][prompt_length:]
    decoded = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    predicted = decoded.split("\n")[0].strip()

    pred_labels.append(predicted)
    if predicted == true_label_name:
        correct += 1

    if (idx + 1) % 100 == 0 or idx == total - 1:
        print(f"  Progress: {len(true_labels)}/{total} (acc: {correct/len(true_labels):.4f})")

accuracy = correct / total
print(f"\n{'='*60}")
print(f"FINAL ACCURACY: {accuracy:.4f} ({correct}/{total})")
print(f"{'='*60}")


Both `max_new_tokens` (=16) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Evaluating on 1519 test samples...



/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

  Progress: 38/1519 (acc: 0.9737)


Both `max_new_tokens` (=16) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

...

  Progress: 1221/1519 (acc: 0.9500)


Both `max_new_tokens` (=16) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

  Progress: 1292/1519 (acc: 0.9497)


Both `max_new_tokens` (=16) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=16) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene


FINAL ACCURACY: 0.9467 (1438/1519)


In [ ]:
unique_labels = sorted(set(true_labels + pred_labels))
report = classification_report(true_labels, pred_labels, labels=unique_labels, zero_division=0)
print("Classification Report:")
print(report)

results = {"accuracy": accuracy, "report": report}
with open(f"{SAVE_DIR}/eval_results.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"Results saved to {SAVE_DIR}/eval_results.json")

Classification Report:
                                                  precision    recall  f1-score   support

                                activate_my_card       1.00      0.97      0.99        40
                                       age_limit       1.00      1.00      1.00        30
                                     atm_support       0.96      0.96      0.96        25
                                automatic_top_up       1.00      0.91      0.95        33
         balance_not_updated_after_bank_transfer       0.90      0.88      0.89        42
balance_not_updated_after_cheque_or_cash_deposit       0.96      0.98      0.97        44
                            card_about_to_expire       0.94      0.91      0.93        34
                                 card_acceptance       0.95      0.95      0.95        20
                                    card_arrival       0.90      0.95      0.93        39
                          card_delivery_estimate       0.93      0.83      0

## 9. Demo Inference

In [ ]:
def predict_intent(message):
    prompt = INFERENCE_TEMPLATE.format(message=message.strip())
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=16,
            do_sample=False,
            use_cache=True,
        )

    prompt_length = inputs["input_ids"].shape[1]
    new_tokens = outputs[0][prompt_length:]
    decoded = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return decoded.split("\n")[0].strip()

examples = [
    "I am still waiting on my card?",
    "How do I change my PIN number?",
    "I was charged twice for the same transaction.",
    "Can I transfer money to another currency?",
    "My payment was declined at the store.",
    "I need to cancel my card immediately.",
    "What are the fees for international transfers?",
]

print("=" * 60)
print("INFERENCE DEMO")
print("=" * 60)
for msg in examples:
    label = predict_intent(msg)
    print(f"\n  Message: {msg}")
    print(f"  Intent:  {label}")

Both `max_new_tokens` (=32) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFERENCE DEMO


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=32) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



  Message: I am still waiting on my card?
  Intent:  card_arrival


Both `max_new_tokens` (=32) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


  Message: Can I transfer money to another currency?
  Intent:  exchange_rate


Both `max_new_tokens` (=32) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



  Message: My payment was declined at the store.
  Intent:  reverted_card_payment?


Both `max_new_tokens` (=32) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



  Message: I need to cancel my card immediately.
  Intent:  card_not_working

  Message: What are the fees for international transfers?
  Intent:  exchange_charge


## 10. Class IntentClassification (theo yêu cầu đề bài)

Cell này định nghĩa class `IntentClassification` với 2 method `__init__` và `__call__` đúng theo format đề bài yêu cầu.

In [ ]:
import yaml

inference_config = {
    "checkpoint_path": "/content/drive/MyDrive/banking-intent-unsloth/outputs",
    "label_map_path": "/content/drive/MyDrive/banking-intent-unsloth/outputs/label_map.json",
    "max_seq_length": 256,
    "load_in_4bit": True,
    "max_new_tokens": 16,
}

with open("inference_config.yaml", "w") as f:
    yaml.dump(inference_config, f)

print("Created inference_config.yaml")

✅ Created inference_config.yaml


In [ ]:
class IntentClassification:
    def __init__(self, model_path):
        import json
        import yaml
        from unsloth import FastLanguageModel

        with open(model_path, "r") as f:
            self.config = yaml.safe_load(f)

        with open(self.config["label_map_path"], "r") as f:
            self.label_map = json.load(f)

        self.valid_labels = set(self.label_map.values())

        self.model, self.tokenizer = FastLanguageModel.from_pretrained(
            model_name=self.config["checkpoint_path"],
            max_seq_length=self.config["max_seq_length"],
            load_in_4bit=self.config.get("load_in_4bit", True),
            dtype=None,
        )
        FastLanguageModel.for_inference(self.model)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

    def __call__(self, message):
        import torch
        prompt = INFERENCE_TEMPLATE.format(message=message.strip())
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

        max_new_tokens = self.config.get("max_new_tokens", 16)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                use_cache=True,
            )

        prompt_length = inputs["input_ids"].shape[1]
        new_tokens = outputs[0][prompt_length:]
        decoded = self.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        predicted_label = decoded.split("\n")[0].strip()

        return predicted_label

print("✅ IntentClassification class has been defined")

==((====))==  Unsloth 2026.4.6: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Both `max_new_tokens` (=32) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


card_arrival


In [ ]:
classifier = IntentClassification("inference_config.yaml")

print("=" * 60)
print("Banking Intent Classifier")
print("Gõ câu hỏi rồi nhấn Enter. Gõ 'quit' để thoát.")
print("=" * 60)

while True:
    message = input("\n> Message: ").strip()
    if message.lower() in ("quit", "exit", "q"):
        print("Goodbye!")
        break
    if not message:
        continue
    label = classifier(message)
    print(f"  → Intent: {label}")

---
## ✅ Hoàn tất!
---